# 02 - Behavior Analysis

## 项目背景

在01中我们生成了用户、对话、事件三张原始数据表。本notebook对这些行为数据进行多维度分析，回答产品层面最核心的问题：产品整体质量如何？哪些场景做得好、哪些做得差？用户留存怎么样？趋势在变好还是变坏？

## 本notebook的目标

从行为数据中提取可执行的产品洞察，覆盖四个分析维度：

| 维度 | 核心问题 | 分析方法 |
|------|---------|---------|
| 整体行为分布 | 用户最常做什么操作 | 事件类型占比、含沉默率的完整漏斗 |
| 场景质量拆解 | 哪个意图场景问题最多 | 按intent交叉分析dislike率、retry率、追问率 |
| 用户活跃与留存 | 用户用了之后还会回来吗 | 按用户类型的活跃天数分布、日留存曲线 |
| 时间趋势 | 质量在变好还是变差 | 核心指标的日级时序变化 |

## 流程

**Step 1: 数据加载与预处理**
- 加载三张表，建立统一配色方案（行为类型、意图场景各一套）
- 将事件表与对话表关联，补充沉默对话的统计

**Step 2: 整体行为分布**
- 计算各事件类型占比，注意沉默用户不在事件表中，需要从对话表反推
- 回答：用户最常见的反馈行为是什么？多少对话没有任何反馈？

**Step 3: 按意图场景拆解**
- 对每个意图计算dislike率、retry率、追问率等核心指标
- 交叉对比：哪个场景的用户不满最集中？
- 回答：产品团队应该优先优化哪个场景？

**Step 4: 用户活跃度与留存**
- 按用户类型分析活跃天数分布
- 计算日级留存率（Day 1, Day 7, Day 14, Day 30）
- 回答：不同用户群体的留存表现差异有多大？

**Step 5: 时间趋势**
- 核心指标（日对话量、dislike率、retry率）的日级时序图
- 回答：产品质量随时间是在改善还是恶化？有没有异常波动？

## 输出

交互式Plotly图表，直接在notebook内展示。关键发现汇总在末尾，供05 Insights Summary引用。

# 02 - Behavior Analysis

## Background

In notebook 01 we generated three raw tables: users, conversations, and events. This notebook performs multi-dimensional behavioral analysis to answer the most fundamental product questions: How is overall quality? Which scenarios perform well and which don't? How is user retention? Are things getting better or worse?

## Objective

Extract actionable product insights across four analytical dimensions:

| Dimension | Core Question | Approach |
|-----------|--------------|----------|
| Overall behavior distribution | What do users do most often? | Event type proportions with silent-rate funnel |
| Per-scenario quality breakdown | Which intent has the most issues? | Cross-tabulation of dislike rate, retry rate, follow-up rate by intent |
| User activity & retention | Do users come back? | Active-day distributions by user type, daily retention curves |
| Time trends | Is quality improving or declining? | Daily time-series of key metrics |

## Pipeline

**Step 1: Data Loading & Preprocessing**
- Load all three tables; define unified color schemes for event types and intent categories
- Join events to conversations; back-fill silent conversations not present in the event table

**Step 2: Overall Behavior Distribution**
- Calculate proportions of each event type, accounting for silent conversations (absent from the event table, inferred from the conversation table)
- Key question: What is the most common user feedback behavior? What share of conversations receive no feedback at all?

**Step 3: Per-Scenario Quality Breakdown**
- Compute dislike rate, retry rate, and follow-up rate for each intent category
- Cross-compare: Where is user dissatisfaction most concentrated?
- Key question: Which scenario should the product team prioritize for optimization?

**Step 4: User Activity & Retention**
- Analyze active-day distributions segmented by user type
- Calculate daily retention rates (Day 1, Day 7, Day 14, Day 30)
- Key question: How different is retention across user segments?

**Step 5: Time Trends**
- Daily time-series of key metrics (conversation volume, dislike rate, retry rate)
- Key question: Is product quality improving or degrading over time? Any anomalous spikes?

## Output

Interactive Plotly charts rendered inline. Key findings are summarized at the end for reference in notebook 05 (Insights Summary).

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook'

# 统一配色方案
COLORS = {
    'like': '#2ecc71',
    'dislike': '#e74c3c',
    'retry': '#f39c12',
    'followup': '#3498db',
    'silent': '#95a5a6'
}

INTENT_COLORS = {
    'code_generation': '#6366f1',
    'knowledge_qa': '#ec4899',
    'creative_writing': '#14b8a6',
    'translation': '#f97316',
    'chitchat': '#8b5cf6',
    'data_analysis': '#06b6d4'
}

# 行为类型中文映射
ACTION_CN = {
    'like': '点赞',
    'dislike': '点踩',
    'retry': '重新生成',
    'followup': '追问',
    'silent': '沉默'
}

# 意图场景中文映射
INTENT_CN = {
    'code_generation': '代码生成',
    'knowledge_qa': '知识问答',
    'creative_writing': '创意写作',
    'translation': '翻译',
    'chitchat': '闲聊',
    'data_analysis': '数据分析'
}

# 用户类型中文映射
UTYPE_CN = {
    'heavy': '重度用户',
    'casual': '轻度用户',
    'churned': '流失用户'
}

# 加载数据
users = pd.read_csv('../data/users.csv', parse_dates=['signup_date'])
convs = pd.read_csv('../data/conversations.csv', parse_dates=['timestamp'])
events = pd.read_csv('../data/events.csv', parse_dates=['event_timestamp'])

print(f'用户: {len(users)} | 对话: {len(convs)} | 事件: {len(events)}')

In [ ]:
# === 整体行为分布（含沉默） ===

# 找出没有任何事件的对话 = 沉默
conv_with_events = events['conversation_id'].unique()
silent_count = len(convs[~convs['conversation_id'].isin(conv_with_events)])

# 统计各事件类型（取每条对话的第一个事件作为"主行为"）
first_events = events.sort_values('event_timestamp').groupby('conversation_id').first().reset_index()
action_counts = first_events['event_type'].value_counts()
action_counts['silent'] = silent_count

# 计算占比
total = action_counts.sum()
action_pcts = (action_counts / total * 100).round(1)

# 画图
action_pcts_cn = action_pcts.rename(index=ACTION_CN)

fig = go.Figure(go.Bar(
    x=action_pcts_cn.index,
    y=action_pcts_cn.values,
    marker_color=[COLORS.get(a, '#999') for a in action_pcts.index],
    text=[f'{v}%' for v in action_pcts_cn.values],
    textposition='outside'
))

fig.update_layout(
    title='整体行为分布（含沉默）',
    xaxis_title='行为类型',
    yaxis_title='占总对话百分比 (%)',
    template='plotly_white',
    height=450,
    showlegend=False,
    yaxis=dict(range=[0, action_pcts_cn.max() * 1.15])
)

fig.show()

In [ ]:
# === 按意图场景拆解 ===

# 把主行为关联回对话表
first_events_with_intent = first_events.merge(
    convs[['conversation_id', 'intent']], on='conversation_id'
)

# 沉默对话也要算进来
silent_convs = convs[~convs['conversation_id'].isin(conv_with_events)][['conversation_id', 'intent']].copy()
silent_convs['event_type'] = 'silent'

combined = pd.concat([
    first_events_with_intent[['conversation_id', 'intent', 'event_type']],
    silent_convs
])

# 计算每个意图下各行为的占比
cross = pd.crosstab(combined['intent'], combined['event_type'], normalize='index') * 100
cross = cross.round(1)

# 按dislike率降序排列
cross = cross.sort_values('dislike', ascending=False)

# 堆叠柱状图
cross_cn = cross.rename(index=INTENT_CN, columns=ACTION_CN)

action_order_cn = ['点赞', '点踩', '重新生成', '追问', '沉默']
fig = go.Figure()

for action_en, action_cn in ACTION_CN.items():
    if action_cn in cross_cn.columns:
        fig.add_trace(go.Bar(
            name=action_cn,
            x=cross_cn.index,
            y=cross_cn[action_cn],
            marker_color=COLORS[action_en],
            text=[f'{v:.0f}%' for v in cross_cn[action_cn]],
            textposition='inside',
            textfont=dict(size=10)
        ))

fig.update_layout(
    barmode='stack',
    title='各意图场景行为分布',
    xaxis_title='意图场景',
    yaxis_title='百分比 (%)',
    template='plotly_white',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)
fig.show()

# 打印关键指标表
print('\n各场景关键指标:')
summary = cross[['dislike', 'retry', 'followup', 'like', 'silent']].copy()
summary['不满意率'] = summary['dislike'] + summary['retry']
summary = summary.sort_values('不满意率', ascending=False)
summary = summary.rename(index=INTENT_CN, columns=ACTION_CN)
print(summary.to_string())

In [ ]:
# === 用户活跃度分布 ===

# 计算每个用户的活跃天数
convs['date'] = convs['timestamp'].dt.date
user_active_days = convs.groupby('user_id')['date'].nunique().reset_index()
user_active_days.columns = ['user_id', 'active_days']
user_active_days = user_active_days.merge(users[['user_id', 'user_type']], on='user_id')

# 箱线图
user_active_days['user_type_cn'] = user_active_days['user_type'].map(UTYPE_CN)

fig = px.box(
    user_active_days,
    x='user_type_cn',
    y='active_days',
    color='user_type_cn',
    color_discrete_map={
        '重度用户': '#6366f1',
        '轻度用户': '#f97316',
        '流失用户': '#ef4444'
    },
    category_orders={'user_type_cn': ['重度用户', '轻度用户', '流失用户']}
)

fig.update_layout(
    title='各用户类型活跃天数分布',
    xaxis_title='用户类型',
    yaxis_title='活跃天数（共31天）',
    template='plotly_white',
    height=450,
    showlegend=False
)

fig.show()

# 打印统计
print('各用户类型活跃天数统计:')
print(user_active_days.groupby('user_type')['active_days'].describe()[['mean', '50%', 'min', 'max']].round(1))

In [ ]:
# === 用户留存分析 ===
DATE_END = pd.Timestamp('2025-03-31')
# 每个用户的首次活跃日期
first_active = convs.groupby('user_id')['date'].min().reset_index()
first_active.columns = ['user_id', 'first_date']

# 每个用户每天是否活跃
user_dates = convs.groupby('user_id')['date'].apply(set).reset_index()
user_dates.columns = ['user_id', 'active_dates']
user_dates = user_dates.merge(first_active, on='user_id')

# 计算各天留存率
retention_days = list(range(0, 31))
retention_rates = []

for d in retention_days:
    retained = 0
    eligible = 0
    for _, row in user_dates.iterrows():
        target_date = row['first_date'] + pd.Timedelta(days=d)
        # 只统计有机会留存到第d天的用户（首次活跃日+d天不超过观察期）
        if target_date <= DATE_END.date():
            eligible += 1
            if target_date in row['active_dates']:
                retained += 1
    rate = retained / eligible * 100 if eligible > 0 else 0
    retention_rates.append(rate)

retention_df = pd.DataFrame({
    '天数': retention_days,
    '留存率': retention_rates
})

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=retention_df['天数'],
    y=retention_df['留存率'],
    mode='lines+markers',
    marker=dict(size=6, color='#6366f1'),
    line=dict(width=2.5, color='#6366f1'),
    fill='tozeroy',
    fillcolor='rgba(99, 102, 241, 0.1)'
))

# 标注关键留存点
for d in [1, 7, 14, 30]:
    if d < len(retention_rates):
        fig.add_annotation(
            x=d, y=retention_rates[d],
            text=f'Day {d}: {retention_rates[d]:.1f}%',
            showarrow=True, arrowhead=2,
            ax=0, ay=-30,
            font=dict(size=11)
        )

fig.update_layout(
    title='用户留存曲线',
    xaxis_title='首次活跃后天数',
    yaxis_title='留存率 (%)',
    template='plotly_white',
    height=450,
    yaxis=dict(range=[0, 105])
)

fig.show()

print(f'Day 1 留存: {retention_rates[1]:.1f}%')
print(f'Day 7 留存: {retention_rates[7]:.1f}%')
print(f'Day 14 留存: {retention_rates[14]:.1f}%')
print(f'Day 30 留存: {retention_rates[30]:.1f}%')


In [ ]:
# === 核心指标日趋势 ===

# 每日对话量
daily_convs = convs.groupby('date').size().reset_index(name='对话量')

# 每日各行为数量
events_with_date = events.merge(convs[['conversation_id', 'timestamp']], on='conversation_id')
events_with_date['date'] = events_with_date['timestamp'].dt.date
daily_actions = events_with_date.groupby(['date', 'event_type']).size().unstack(fill_value=0).reset_index()

# 合并，计算日级指标
daily = daily_convs.merge(daily_actions, on='date', how='left').fillna(0)
daily['点踩率'] = daily['dislike'] / daily['对话量'] * 100
daily['重新生成率'] = daily['retry'] / daily['对话量'] * 100
daily['点赞率'] = daily['like'] / daily['对话量'] * 100

# 三条线画在一起
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=('每日对话量', '每日关键指标趋势'))

# 上图：每日对话量
fig.add_trace(go.Bar(
    x=daily['date'], y=daily['对话量'],
    marker_color='rgba(99, 102, 241, 0.4)',
    name='对话量'
), row=1, col=1)

# 下图：三条指标线
for col, color, name in [
    ('点赞率', '#2ecc71', '点赞率'),
    ('点踩率', '#e74c3c', '点踩率'),
    ('重新生成率', '#f39c12', '重新生成率')
]:
    fig.add_trace(go.Scatter(
        x=daily['date'], y=daily[col],
        mode='lines+markers',
        marker=dict(size=4),
        line=dict(width=2, color=color),
        name=name
    ), row=2, col=1)

fig.update_layout(
    title='每日核心指标趋势',
    template='plotly_white',
    height=600,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)

fig.update_yaxes(title_text='对话量', row=1, col=1)
fig.update_yaxes(title_text='百分比 (%)', row=2, col=1)

fig.show()

## 关键发现

**1. 整体行为分布**
- 沉默率 24.1%，近四分之一的对话没有任何用户反馈
- 点赞率（23.7%）略低于沉默率，说明满意的用户也不一定会主动反馈

**2. 场景质量差异显著**
- 代码生成的不满意率最高（45.1%），主要来自高retry率（30.3%）
- 知识问答的dislike率最高（24.3%），疑似幻觉问题
- 创意写作和翻译表现较好，点赞率均超35%
- 闲聊沉默率高达65%，用户参与度极低

**3. 用户分群差异明显**
- 重度用户月活跃约25天，轻度用户约8天，流失用户仅2天
- 三类用户的活跃度几乎没有重叠，分群有效

**4. 时间趋势平稳**
- 日对话量稳定在700左右，无明显增长或下降
- 点踩率和retry率日间波动小，暂无异常事件

> 下一步：03 将对 bad case 进行语义归因分析，回答"为什么不满意"